# 07. シミュレーションとモデリング — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

シミュレーションとモデリングは、因果仮説を数式モデルに落とし込み、不確実性を正面から扱って定量的に検証する手法である。本ノートブックでは、企業の生成AIツール採用率を Bass 拡散モデルで表現し、革新係数 p と模倣係数 q を確率分布からサンプリングするモンテカルロ（5000試行）で、各年採用率の中央値と信頼帯、採用ピーク年の分布を求める。

必要なライブラリを読み込む。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


離散時間 Bass 拡散モデルを定義する。各期の新規採用率を `(p + q·F) · (1 − F)` で与え、累積採用率 F の年次系列を返す。

In [ ]:
def bass_diffusion(p, q, n_years):
    """離散時間 Bass 拡散モデル。累積採用率 F の年次系列を返す。"""
    F = np.zeros(n_years + 1)          # F[t] : t年末の累積採用率（0-1）
    adopt = np.zeros(n_years + 1)      # adopt[t] : t年の新規採用率
    for t in range(1, n_years + 1):
        # 新規採用 = (革新による採用 + 模倣による採用) × 未採用者割合
        new = (p + q * F[t - 1]) * (1.0 - F[t - 1])
        adopt[t] = new
        F[t] = min(F[t - 1] + new, 1.0)
    return F, adopt

p と q を 1 点に固定せず確率分布からサンプリングし、各サンプルで拡散曲線を計算するモンテカルロ関数を定義する。

In [ ]:
def monte_carlo(n_trials, n_years, start_year, rng):
    """p,q を分布からサンプリングし拡散をモンテカルロ実行。"""
    # 生成AIツールは口コミ・FOMO・低い導入障壁により模倣が強い
    # → 革新係数 p は中庸、模倣係数 q を高め（平均0.4前後）に設定する。
    #   採用が量子クラウドより速く立ち上がる worked example。
    p_samples = np.clip(rng.normal(0.030, 0.010, n_trials), 0.005, None)
    q_samples = np.clip(rng.normal(0.40, 0.10, n_trials), 0.10, None)

    all_F = np.zeros((n_trials, n_years + 1))
    peak_years = np.zeros(n_trials, dtype=int)
    for i in range(n_trials):
        F, adopt = bass_diffusion(p_samples[i], q_samples[i], n_years)
        all_F[i] = F
        peak_years[i] = start_year + int(np.argmax(adopt[1:])) + 1
    return all_F, peak_years, p_samples, q_samples

モンテカルロを 5000 試行実行し、各年採用率の中央値と 5-95% 信頼帯を求める。

In [ ]:
rng = np.random.default_rng(42)
n_trials, n_years, start_year = 5000, 20, 2025

all_F, peak_years, p_s, q_s = monte_carlo(n_trials, n_years,
                                          start_year, rng)

# --- 各年採用率の中央値と信頼帯 ---
median = np.median(all_F, axis=0)
lo = np.percentile(all_F, 5, axis=0)
hi = np.percentile(all_F, 95, axis=0)
years = np.arange(start_year, start_year + n_years + 1)

print("=" * 62)
print("Bass拡散モデル : 企業の生成AIツール採用率（モンテカルロ5000試行）")
print("=" * 62)
print(f"{'年':<8}{'中央値':>10}{'5%':>10}{'95%':>10}")
print("-" * 62)
for idx in range(0, n_years + 1, 4):
    print(f"{years[idx]:<8}{median[idx]*100:>9.1f}%"
          f"{lo[idx]*100:>9.1f}%{hi[idx]*100:>9.1f}%")

2040年の累積採用率を要約する。

In [ ]:
i2040 = 2040 - start_year
print(f"[2040年の累積採用率] 中央値 {median[i2040]*100:.1f}% / "
      f"90%信頼帯 {lo[i2040]*100:.1f}-{hi[i2040]*100:.1f}%")

採用ピーク年（新規採用率が最大となる年）の分布を集計する。

In [ ]:
print("--- 採用ピーク年の分布 ---")
print(f"  中央値      : {int(np.median(peak_years))} 年")
print(f"  5-95%範囲   : {np.percentile(peak_years,5):.0f}"
      f"-{np.percentile(peak_years,95):.0f} 年")
vals, counts = np.unique(peak_years, return_counts=True)
for v, c in zip(vals, counts):
    if c / n_trials >= 0.03:  # 3%以上の年のみ表示
        bar = "#" * int(c / n_trials * 100)
        print(f"  {v} | {bar} {c/n_trials*100:.1f}%")

print()
print("[解釈] 模倣係数 q が高いため採用は早期に立ち上がり、")
print("       ピーク年は量子クラウドより前倒しになる。それでも")
print("       ピーク年に幅が残ること自体が、AI投資判断の")
print("       不確実性を示している。")

## 可視化: モンテカルロ採用曲線

左図に各年採用率の中央値の折れ線と 5-95% 信頼帯（fill_between）を、右図に採用ピーク年のヒストグラムを描く。信頼帯の幅・ピーク年の散らばりが予測の不確実性を表す。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左: 採用曲線の中央値と信頼帯
ax = axes[0]
ax.fill_between(years, lo * 100, hi * 100, alpha=0.3, color="steelblue",
                label="90% interval")
ax.plot(years, median * 100, color="navy", lw=2, label="median")
ax.set_xlabel("Year")
ax.set_ylabel("Cumulative adoption rate (%)")
ax.set_title("Enterprise generative-AI adoption (Bass diffusion, Monte Carlo)")
ax.legend()
ax.grid(alpha=0.3)

# 右: 採用ピーク年のヒストグラム
ax = axes[1]
bins = np.arange(peak_years.min() - 0.5, peak_years.max() + 1.5, 1)
ax.hist(peak_years, bins=bins, color="darkorange", edgecolor="white")
ax.axvline(np.median(peak_years), color="black", ls="--",
           label=f"median {int(np.median(peak_years))}")
ax.set_xlabel("Adoption peak year")
ax.set_ylabel("Number of trials")
ax.set_title("Distribution of adoption peak year")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## パラメータ感度分析（トルネード図）\n\nモンテカルロは p・q を同時に分布から振り、結果のばらつき全体を示した。一方で「どのパラメータが結果を最も左右しているか」は同時サンプリングからは切り分けにくい。そこで感度分析では、パラメータを **一つずつ** 低位/高位に振り、他は基準値に固定して Bass モデルを決定論的に走らせ、各パラメータ単独の影響度を測る。\n\nここでは革新係数 p、模倣係数 q に加え、市場規模 m（最大採用企業数。採用企業数 = m × 累積採用率 F）も対象とする。成果指標は 2040 年の累積採用企業数とする。

In [ ]:
# --- 感度分析: 各パラメータの基準値・低位値・高位値を定める ---
# モンテカルロで使った分布の中心値を基準値とし、+-30% で低位/高位を振る。
# 市場規模 m は累積採用率に対する係数（採用企業数 = m * F）。
SENS_YEAR = 2040          # 注目する成果指標を測る年次
i_eval = SENS_YEAR - start_year

base = {"p": 0.030, "q": 0.40, "m": 100000}   # 基準値
SPREAD = 0.30             # 低位/高位の振れ幅（基準値からの割合）

param_ranges = {}
for name, val in base.items():
    param_ranges[name] = {"low": val * (1 - SPREAD),
                          "base": val,
                          "high": val * (1 + SPREAD)}

print("=" * 58)
print("感度分析パラメータ設定（基準値 +-30%）")
print("=" * 58)
print(f"{'param':<8}{'low':>14}{'base':>14}{'high':>14}")
for name, r in param_ranges.items():
    print(f"{name:<8}{r['low']:>14.4f}{r['base']:>14.4f}{r['high']:>14.4f}")


各パラメータを単独で低位/高位に振り、2040年の累積採用企業数の変化を測る。他のパラメータは基準値に固定する（一要因感度分析）。

In [ ]:
# --- 一要因ずつ振って成果指標を決定論的に評価する ---
def outcome(p, q, m):
    """与えたパラメータで Bass モデルを決定論的に走らせ、
    SENS_YEAR 時点の累積採用企業数（= m * F）を返す。"""
    F, adopt = bass_diffusion(p, q, n_years)
    return m * F[i_eval]

# 全パラメータ基準値での成果（トルネード図の中心線）
base_outcome = outcome(base["p"], base["q"], base["m"])

# 各パラメータを単独で low / high に振る（他は基準値に固定）
results = {}
for name in base:
    args_low = dict(base);  args_low[name]  = param_ranges[name]["low"]
    args_high = dict(base); args_high[name] = param_ranges[name]["high"]
    out_low = outcome(args_low["p"], args_low["q"], args_low["m"])
    out_high = outcome(args_high["p"], args_high["q"], args_high["m"])
    results[name] = {"low": out_low, "high": out_high,
                     "swing": abs(out_high - out_low)}

# 振れ幅（swing）の大きい順に並べる
order = sorted(results, key=lambda n: results[n]["swing"])

print("=" * 64)
print(f"{SENS_YEAR}年 累積採用企業数に対する一要因感度（基準={base_outcome:,.0f}）")
print("=" * 64)
print(f"{'param':<8}{'low':>14}{'high':>14}{'swing':>14}")
for name in reversed(order):
    r = results[name]
    print(f"{name:<8}{r['low']:>14,.0f}{r['high']:>14,.0f}{r['swing']:>14,.0f}")


各パラメータの低位・高位での成果値を、基準値を中心とした横棒で並べる。振れ幅の大きいパラメータほど上に来るよう並べ替える。

In [ ]:
# --- トルネード図: 各パラメータの振れ幅を横棒で描く ---
fig, ax = plt.subplots(figsize=(9, 4.5))

y_pos = np.arange(len(order))
for i, name in enumerate(order):
    r = results[name]
    # 基準値からの振れ: low 側と high 側に分けて横棒を描く
    left = min(r["low"], r["high"])
    right = max(r["low"], r["high"])
    ax.barh(i, base_outcome - left, left=left, height=0.6,
            color="indianred", edgecolor="white",
            label="low-side" if i == 0 else None)
    ax.barh(i, right - base_outcome, left=base_outcome, height=0.6,
            color="steelblue", edgecolor="white",
            label="high-side" if i == 0 else None)

ax.axvline(base_outcome, color="black", lw=1.5, ls="--",
           label="base case")
ax.set_yticks(y_pos)
ax.set_yticklabels([n.upper() for n in order])
ax.set_xlabel(f"Cumulative adopters in {SENS_YEAR}")
ax.set_title("Tornado diagram: one-at-a-time parameter sensitivity")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

top = order[-1]
print(f"[結論] 成果指標を最も大きく動かすパラメータ: {top.upper()} "
      f"(swing={results[top]['swing']:,.0f})")


### 考察\n\nトルネード図は上に行くほど振れ幅が大きく、生成AIツールの 2040 年採用企業数という結論が **どの不確実性に最も依存しているか** を一目で示す。振れ幅が最大のパラメータは、データ収集や専門家ヒアリングで優先的に精度を高めるべき対象である。逆に振れ幅の小さいパラメータは多少誤っても結論を大きく揺らさない。\n\n**モンテカルロと感度分析の役割の違い**: モンテカルロ（同時分布）は「結論にどれだけ幅があるか」を示すのに対し、感度分析（一要因ずつ）は「その幅が **どのパラメータ由来か** 」を分解する。両者は補完関係にあり、前者でリスクの大きさを把握し、後者でリスク低減の打ち手を特定する。

## 未来デザイン論文での使われ方と結論への影響

未来デザインの論文においてシミュレーションとモデリングは、対象システムの動的な振る舞いを数式とパラメータで記述し、時系列の予測や感度分析を提示するために用いられる。論文の典型的な使われ方は、採用や普及、資源消費といった量の将来軌道を計算し、信頼区間や複数シナリオの帯を添えて「この条件ならこの範囲に収まる」と論じることである。したがってこの手法が生み出す結論は、定量的な軌道とその不確実性の幅という型をとり、数値の説得力をもって提示される。

結論の型は具体的な数値だが、その数値はモデル構造とパラメータの選択の産物である点に注意が要る。境界設定の面では、モデルに変数として組み込まれた要素しか結論に反映されず、境界の外に置かれた要因は存在しないものとして扱われる。時間観の面では、未来は方程式が記述する力学に従って展開すると仮定され、構造が安定している限り未来は計算可能なものとみなされる。価値の所在は、何を変数とし何を定数とするか、どの関数形を選ぶかという定式化の段階に埋め込まれ、そこに分析者の世界観が静かに入り込む。

最大の限界は、見かけの精度（false precision）である。出力が小数点以下まで提示されると、その精度はモデルの妥当性を保証するかのように見えるが、実際には入力の不確実性と構造の恣意性を覆い隠しているにすぎない。モデル境界の外で起きる構造変化——制度の転換、想定外の競合技術、力学そのものの変質——は定義上結論に現れない。したがってこの手法に依拠した論文の結論は、提示された数値そのものよりも、その数値を支える境界とパラメータの仮定をこそ吟味対象として読むべきものである。

## 発展課題

**課題A**: p と q をそれぞれ単独で「低位/基準/高位」に振り、2040年採用率の変化幅を棒で並べたトルネード図を作成せよ。どちらの係数が結果を支配するかを可視化する（感度分析）。

**課題B**: 採用率だけでなく「導入企業1社あたりの生産性向上効果」にも不確実性を与え、総効果 = 採用企業数 × 1社あたり効果 の分布を求めよ。採用率の不確実性と効果単価の不確実性が掛け合わさる効果を観察する。